In [0]:
# Databricks notebook source
# Integration tests for the upload pipeline
#
# These require a live Databricks workspace with the manifest and bronze
# tables present. They are NOT run by CI — run them by hand after
# changing anything in the Spark paths.
#
# Each test cleans up after itself. Run cells in order the first time.

In [0]:
# Cell 1 — Setup
%pip install pytest

In [0]:
dbutils.library.restartPython()

In [0]:
# imports and helpers

import os
import shutil
import sys
import tempfile
import uuid

BASE = "/Workspace/Repos/ibrahimothmanre@gmail.com/ea_data_mgmt"
if BASE not in sys.path:
    sys.path.insert(0, BASE)

from pyspark.sql import functions as F

from ea_pipeline.bronze import load_upload_to_bronze
from ea_pipeline.config import BRONZE_TABLES, DATASET_DIRECTORIES, MANIFEST_TABLE, spark
from ea_pipeline.register import register_uploaded_file
from ea_pipeline.states import UploadStatus
from ea_pipeline.validate import validate_registered_upload


def make_test_upload(dataset_name, csv_content, validate=True):
    """
    Create a CSV in the landing directory, register it, optionally
    validate it. Returns the upload_id.

    The file name is unique per call, because registration refuses a
    repeated path.
    """
    file_name = f"itest_{dataset_name}_{uuid.uuid4().hex[:8]}.csv"
    target_path = os.path.join(DATASET_DIRECTORIES[dataset_name], file_name)

    # Write via a temp file so a half-written file never appears in the
    # landing directory.
    with tempfile.NamedTemporaryFile(
        mode="w", encoding="utf-8", newline="", suffix=".csv", delete=False
    ) as temp_file:
        temp_file.write(csv_content)
        temp_name = temp_file.name

    shutil.move(temp_name, target_path)

    upload_id = register_uploaded_file(
        file_path=target_path,
        dataset_name=dataset_name,
        source_system="TEST",
        reporting_period="2026-Q3",
        verbose=False,
    )

    if validate:
        validate_registered_upload(upload_id, verbose=False)

    return upload_id


def set_status(upload_id, status, minutes_ago=0):
    """
    Force a manifest row into a given state.

    Deliberately bypasses the safeguards, to reach states the code will
    not produce on purpose. Test-only.
    """
    spark.sql(
        f"""
        UPDATE {MANIFEST_TABLE}
        SET status = :status,
            updated_at = current_timestamp() - INTERVAL {minutes_ago} MINUTES
        WHERE upload_id = :upload_id
        """,
        args={"status": status, "upload_id": upload_id},
    )


def manifest_row(upload_id):
    return (
        spark.table(MANIFEST_TABLE)
        .where(F.col("upload_id") == upload_id)
        .first()
    )


def bronze_count(dataset_name, upload_id):
    return (
        spark.table(BRONZE_TABLES[dataset_name])
        .where(F.col("upload_id") == upload_id)
        .count()
    )


def cleanup(dataset_name, upload_id):
    """Remove a test upload from bronze, the manifest, and disk."""
    record = manifest_row(upload_id)

    spark.sql(
        f"DELETE FROM {BRONZE_TABLES[dataset_name]} WHERE upload_id = :id",
        args={"id": upload_id},
    )
    spark.sql(
        f"DELETE FROM {MANIFEST_TABLE} WHERE upload_id = :id",
        args={"id": upload_id},
    )

    if record and os.path.isfile(record["stored_file_path"]):
        os.remove(record["stored_file_path"])


print("helpers ready")

In [0]:
# test data
CLEAN = (
    "project_id,project_name,project_status\n"
    "P0011,Website Rebuild,Active\n"
    "P0022,Mobile App,Active\n"
    "P0033,Data Platform,Closed\n"
)

EXCEL_HEADERS = (
    "Project ID,Project Name,Project Status\n"
    "P001,Website Rebuild,Active\n"
    "P002,Mobile App,Active\n"
)

EXTRA_COLUMN = (
    "project_id,project_name,project_status,project_region\n"
    "P001,Website Rebuild,Active,EMEA\n"
    "P002,Mobile App,Active,APAC\n"
)

# 2 bad rows out of 100 — under the 5% threshold
MOSTLY_GOOD = (
    "project_id,project_name,project_status\n"
    + "".join(f"P{i:03d},Project {i},Active\n" for i in range(98))
    + "P098,Bad Row, Extra Value,Active\n"
    + "P099,Another Bad, Extra,Active\n"
)

# 10 bad rows out of 20 — well over the threshold
MOSTLY_BAD = (
    "project_id,project_name,project_status\n"
    + "".join(f"P{i:03d},Project {i},Active\n" for i in range(10))
    + "".join(f"P{i:03d},Bad Row, Extra Value,Active\n" for i in range(10, 20))
)

MULTILINE = (
    "project_id,project_name,project_status\n"
    'P001,"Website Rebuild\nPhase 2",Active\n'
    "P002,Mobile App,Active\n"
)

print("fixtures ready")

In [0]:
# T1: happy path
upload_id = make_test_upload("projects", CLEAN)

try:
    outcome = load_upload_to_bronze(upload_id, verbose=False)
    row = manifest_row(upload_id)

    assert outcome.status == UploadStatus.PROCESSED, outcome.status
    assert outcome.row_count == 3, outcome.row_count
    assert outcome.corrupt_row_count == 0
    assert row["bronze_row_count"] == 3
    assert row["bronze_processed_at"] is not None
    assert bronze_count("projects", upload_id) == 3

    # Validation counted the same rows Spark did
    assert row["source_row_count"] == 3, row["source_row_count"]

    print("T1 PASS — clean file loads")
finally:
    cleanup("projects", upload_id)

In [0]:
# T2: Excel-style headers
upload_id = make_test_upload("projects", EXCEL_HEADERS)

try:
    outcome = load_upload_to_bronze(upload_id, verbose=False)

    assert outcome.status == UploadStatus.PROCESSED

    # Bronze column names must match the normalised contract names
    columns = spark.table(BRONZE_TABLES["projects"]).columns
    assert "project_id" in columns
    assert "Project ID" not in columns

    loaded = (
        spark.table(BRONZE_TABLES["projects"])
        .where(F.col("upload_id") == upload_id)
        .first()
    )
    assert loaded["project_id"] == "P001", loaded["project_id"]

    print("T2 PASS — Excel headers normalised end to end")
finally:
    cleanup("projects", upload_id)

In [0]:
# T4: running twice does not duplicate
upload_id = make_test_upload("projects", CLEAN)

try:
    load_upload_to_bronze(upload_id, verbose=False)

    # Second call must be refused: the row is already PROCESSED
    try:
        load_upload_to_bronze(upload_id, verbose=False)
        raise AssertionError("second load should have been refused")
    except ValueError as error:
        assert "PROCESSED" in str(error), str(error)

    assert bronze_count("projects", upload_id) == 3

    print("T4 PASS — re-running a PROCESSED upload is refused")
finally:
    cleanup("projects", upload_id)

In [0]:
# T5: partial previous load is cleared
upload_id = make_test_upload("projects", CLEAN)

try:
    load_upload_to_bronze(upload_id, verbose=False)
    assert bronze_count("projects", upload_id) == 3

    # Pretend the previous run died before recording PROCESSED
    set_status(upload_id, "VALIDATED")

    load_upload_to_bronze(upload_id, verbose=False)

    # 3, not 6 — _clear_previous_load did its job
    assert bronze_count("projects", upload_id) == 3, bronze_count("projects", upload_id)

    print("T5 PASS — partial load cleared, no duplication")
finally:
    cleanup("projects", upload_id)

In [0]:
# T6: stale claim recovery
upload_id = make_test_upload("projects", CLEAN)

try:
    # Abandoned 45 minutes ago — past the 30 minute window
    set_status(upload_id, "PROCESSING", minutes_ago=45)

    outcome = load_upload_to_bronze(upload_id, verbose=False)
    assert outcome.status == UploadStatus.PROCESSED

    print("T6a PASS — stale PROCESSING claim reclaimed")

    # Now claimed 5 minutes ago — someone may still be working on it
    set_status(upload_id, "PROCESSING", minutes_ago=5)

    try:
        load_upload_to_bronze(upload_id, verbose=False)
        raise AssertionError("a fresh PROCESSING claim should be refused")
    except ValueError:
        pass

    print("T6b PASS — fresh PROCESSING claim refused")
finally:
    cleanup("projects", upload_id)

In [0]:
# T9 — over the threshold, rejected
upload_id = make_test_upload("projects", MOSTLY_BAD)

try:
    outcome = load_upload_to_bronze(upload_id, verbose=False)

    assert outcome.status == UploadStatus.BRONZE_REJECTED, outcome.status
    assert "exceeds" in outcome.message, outcome.message
    assert bronze_count("projects", upload_id) == 0   # nothing written

    print(f"T9 PASS — rejected: {outcome.message}")
finally:
    cleanup("projects", upload_id)

In [0]:
# T10 — under the threshold, loaded with the count recorded
upload_id = make_test_upload("projects", MOSTLY_GOOD)

try:
    outcome = load_upload_to_bronze(upload_id, verbose=False)
    row = manifest_row(upload_id)

    assert outcome.status == UploadStatus.PROCESSED, outcome.status
    assert outcome.corrupt_row_count > 0, "bad rows should have been counted"
    assert row["bronze_corrupt_row_count"] == outcome.corrupt_row_count

    # Corrupt rows are kept, not dropped
    assert bronze_count("projects", upload_id) == outcome.row_count

    print(
        f"T10 PASS — loaded {outcome.row_count} rows, "
        f"{outcome.corrupt_row_count} corrupt"
    )
finally:
    cleanup("projects", upload_id)

In [0]:
# T13: schema evolution
before = set(spark.table(BRONZE_TABLES["projects"]).columns)

upload_id = make_test_upload("projects", EXTRA_COLUMN)

try:
    outcome = load_upload_to_bronze(upload_id, verbose=False)
    assert outcome.status == UploadStatus.PROCESSED, outcome.status

    after = set(spark.table(BRONZE_TABLES["projects"]).columns)

    assert "project_region" in after, after - before

    loaded = (
        spark.table(BRONZE_TABLES["projects"])
        .where(F.col("upload_id") == upload_id)
        .first()
    )
    assert loaded["project_region"] == "EMEA"

    print(f"T13 PASS — mergeSchema added: {after - before}")
finally:
    cleanup("projects", upload_id)

In [0]:
%sql
ALTER TABLE ea_dev.bronze.projects DROP COLUMN project_region;
describe table ea_dev.bronze.projects;

In [0]:
# T15: multiline reconciliation
upload_id = make_test_upload("projects", MULTILINE)

try:
    outcome = load_upload_to_bronze(upload_id, verbose=False)
    row = manifest_row(upload_id)

    # Python's csv and Spark's multiLine reader must agree: 2 rows, not 3
    assert row["source_row_count"] == 2, row["source_row_count"]
    assert outcome.row_count == 2, outcome.row_count

    loaded = (
        spark.table(BRONZE_TABLES["projects"])
        .where(F.col("upload_id") == upload_id)
        .where(F.col("project_id") == "P001")
        .first()
    )
    assert "\n" in loaded["project_name"], repr(loaded["project_name"])

    print("T15 PASS — multiline value preserved, counts agree")
finally:
    cleanup("projects", upload_id)

In [0]:
# T16: status gate
upload_id = make_test_upload("projects", CLEAN, validate=False)   # stays RECEIVED

try:
    for status in ["RECEIVED", "VALIDATING", "REJECTED", "DUPLICATE", "PROCESSED"]:
        set_status(upload_id, status)

        try:
            load_upload_to_bronze(upload_id, verbose=False)
            raise AssertionError(f"{status} should not be loadable")
        except ValueError as error:
            assert status in str(error), str(error)

    print("T11 PASS — all ineligible statuses refused")
finally:
    cleanup("projects", upload_id)